# 🚀 İstanbul Metre - 3-Sınıflı BERTurk İnce Ayar (Fine-Tuning) Defteri

Bu Jupyter Notebook, Google Colab'in ücretsiz **T4 GPU** donanımını kullanarak **`dbmdz/bert-base-turkish-128k-cased`** (BERTurk) modelini, ürettiğimiz **5K dengeli Türkçe tweet veri kümesi** ile 3 sınıflı (Pozitif, Negatif, Nötr) olarak eğitmek için tasarlanmıştır.

---

## 🎯 1. Neden Bu Eğitimi Yapıyoruz?
Türkçe, sondan eklemeli (agglutinative) yapısı nedeniyle NLP modelleri için oldukça zorlayıcı bir dildir. Standart BERT modelleri (32k kelime dağarcığına sahip olanlar), sosyal medyadaki ek almış Türkçe kelimeleri (Örn: `#istanbul'daki`, `zamlanmışlar`) çok fazla parçalayarak modelin cümlenin duygusunu anlamasını zorlaştırır.

Bu projede **128k geniş kelime dağarcığına sahip cased (büyük/küçük harf duyarlı) BERTurk modelini** seçtik. Bu model:
* Türkçe kökleri ve duygu belirten ekleri (örneğin olumsuzluk ekleri, sitem ekleri) tek bir bütün olarak korur.
* Sosyal medyadaki yazım tarzını, büyük/küçük harf vurgularını ve ironileri çok daha yüksek doğrulukla yakalar.

---

## 🎭 2. Duygu Sınıflarımız ve Keskin Karar Kuralları
Eğitimde kullandığımız 5K veri setindeki tweetler, 3 sınıfa ayrılmıştır. Modelimizin öğrenmesi gereken kurallar şunlardır:

* **🔴 Etiket 0: Negatif / İsyan / Sarkastik İroni**
  * *Doğrudan Negatif:* Hayat pahalılığı, fahiş kiralar, zamlar, metrobüs arızaları veya trafik çilesi.
  * *Sarkastik İroni (İğneleme):* Aslında kötü giden bir durumu sahte bir şekilde övüyormuş gibi yapıp sonuna gülücük koyan tweetler (Örn: *"Mazota yine zam gelmiş, şahlanıyoruz maşallah :)"*). Bu tweetler **kesinlikle 0 (Negatif)** olarak etiketlenmiştir.

* **🟡 Etiket 1: Nötr / Resmi / Bilgilendirici**
  * *Duygusuz Veriler:* Tarafsız haber başlıkları, resmi belediye duyuruları, yol durumu güncellemeleri, borsa ve esnaf odası resmi raporları. Kesinlikle hiçbir kişisel görüş veya duygu barındırmaz.

* **🟢 Etiket 2: Pozitif / Gerçek Memnuniyet / Teşekkür**
  * *Doğrudan Pozitif:* Samimi memnuniyetler, dürüst esnaf övgüleri, vapur keyfi paylaşımları, kira kolaylığı sağlayan ev sahiplerine teşekkürler.
  * *Gerçek Mutluluklu Gülücükler:* İroni içermeyen, durumun gerçekten iyi olmasından duyulan samimi sevinç içeren gülücüklü tweetler (Örn: *"İstanbul'da bu sabah vapur keyfi yapıyorum, hava harika, durumlar çok güzel 😊"*). Bunlar **kesinlikle 2 (Pozitif)** olarak etiketlenmiştir.

---

## 📌 3. Google Colab'de Eğitim Adımları
1. **GPU Ayarı:** Üst menüden `Runtime` > `Change runtime type` seçip **T4 GPU**'yu aktif edin (Eğer yapmadıysanız).
2. **Veri Yükleme:** Sol menüdeki dosya simgesine (📂) tıklayıp ürettiğiniz `text,label,reason_5k.txt` dosyasını sürükleyip Colab'e yükleyin.
3. **Defteri Çalıştırın:** Hücreleri sırayla (Shift + Enter) çalıştırarak eğitimi koşturun.
4. **Modeli İndirin:** Eğitim sonunda oluşan `fine_tuned_bert.zip` dosyasını sol menüden sağ tıklayıp bilgisayarınıza indirin ve proje kök dizinine çıkartın!

## 1. Gerekli Kütüphanelerin Kurulması

In [ ]:
!pip install -q transformers[torch] datasets accelerate scikit-learn pandas

## 2. Donanım ve Dosya Kontrolü

In [ ]:
import os
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Cihaz: {device}")
if device.type != 'cuda':
    print("⚠️ UYARI: GPU aktif değil! Lütfen Runtime ayarlarından T4 GPU'yu seçtiğinizden emin olun. Yoksa eğitim saatler sürer!")
else:
    print("✅ Harika! T4 GPU aktif, eğitim çok hızlı tamamlanacak (5-10 dakika).")

TXT_PATH = "text,label,reason_5k.txt"
if not os.path.exists(TXT_PATH):
    print(f"❌ HATA: '{TXT_PATH}' dosyası Colab kök dizininde bulunamadı!")
    print("Lütfen sol taraftaki dosya menüsünden veri setini yükleyin.")
else:
    print(f"✅ '{TXT_PATH}' başarıyla bulundu! Veriler okunmaya hazır.")

## 3. Sınıf Ağırlıkları ve Veri Hazırlığı

In [ ]:
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Veriyi Oku
records = []
with open(TXT_PATH, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        if not row or len(row) < 2:
            continue
        text = row[0].strip()
        label_str = row[1].strip()
        if text.lower() == "text" and label_str.lower() == "label":
            continue
        try:
            label = int(label_str)
            records.append({"text": text, "label": label})
        except ValueError:
            continue
            
print(f"Toplam okunan veri sayısı: {len(records)}")
df = pd.DataFrame(records)

# Sınıf Dağılımını Hesapla
label_counts = df['label'].value_counts().to_dict()
print("Sınıf Dağılımı:")
for lbl, cnt in sorted(label_counts.items()):
    print(f"  Etiket {lbl}: {cnt} adet")
    
# Dengesizlikler için Class Weights hesapla
total = len(df)
class_weights = []
for lbl in [0, 1, 2]:
    count = label_counts.get(lbl, 1)
    class_weights.append(total / (3.0 * count))
print(f"Hesaplanan Sınıf Ağırlıkları (Class Weights): {class_weights}")

# Train/Test Split
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['label'])
print(f"Eğitim Verisi: {len(train_df)} | Doğrulama Verisi: {len(val_df)}")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

## 4. Özel Kayıp Fonksiyonu (Weighted Trainer) Tanımı

In [ ]:
from torch import nn
from transformers import Trainer
import numpy as np

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        if self.class_weights is not None:
            weight_tensor = torch.tensor(self.class_weights, dtype=torch.float).to(model.device)
            loss_fct = nn.CrossEntropyLoss(weight=weight_tensor)
        else:
            loss_fct = nn.CrossEntropyLoss()
            
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = np.mean(predictions == labels)
    
    # Calculate Macro F1
    unique_labels = [0, 1, 2]
    f1_scores = []
    
    for cls in unique_labels:
        tp = np.sum((predictions == cls) & (labels == cls))
        fp = np.sum((predictions == cls) & (labels != cls))
        fn = np.sum((predictions != cls) & (labels == cls))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        f1_scores.append(f1)
        
    macro_f1 = np.mean(f1_scores)
    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }

## 5. Tokenizer ve Model Yükleme

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

MODEL_NAME = "dbmdz/bert-base-turkish-128k-cased"

print(f"Model ve Tokenizer yükleniyor: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)
    
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 6. Eğitim Hiperparametreleri ve Başlatma

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_bert",
    learning_rate=2e-5,
    per_device_train_batch_size=32,   # GPU'da daha büyük batch size seçiyoruz
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=20,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights
)

print("🚀 İnce ayar eğitimi (Fine-Tuning) T4 GPU üzerinde başlatılıyor...")
trainer.train()

## 7. Model Kaydetme ve Zip Formatında İndirme Hazırlığı

In [ ]:
MODEL_DIR = "./fine_tuned_bert"
print(f"Model ve Tokenizer kaydediliyor: {MODEL_DIR}")
os.makedirs(MODEL_DIR, exist_ok=True)
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("📦 Model sıkıştırılıyor... (Zipleme)")
!zip -r fine_tuned_bert.zip fine_tuned_bert/
print("✅ TAMAMLANDI! Sol menüden 'fine_tuned_bert.zip' dosyasını sağ tıklayıp indirerek bilgisayarınıza aktarabilirsiniz.")